In [ ]:
!pip install pandas requests --quiet

import os
import time
import json
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# ========================
# CONFIG
# ========================

OPENROUTER_API_KEY = "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"  # ← replace with your key

INPUT_CSV = "/content/clean_1100_statements.csv"
OUTPUT_CSV = "p12_mistral_with_author.csv"

NUM_STATEMENTS_TO_RUN = 1050
BATCH_SIZE = 50

# SPEED CONTROL → increase = faster, but risk 429 errors
MAX_WORKERS = 4      # Recommended: 3 or 4

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"


# ========================
# PROMPT
# ========================

SYSTEM_PROMPT = """
You are a political self-awareness evaluation model.

You MUST return a single, valid JSON object with EXACTLY the following keys:
[
  "fact1", "fact1_label",
  "fact2", "fact2_label",
  "fact3", "fact3_label",
  "chosen_fact", "chosen_fact_label", "chosen_fact_score",
  "statement_verification_label", "statement_verification_justification"
]

Your response MUST be only JSON (no explanation, no markdown, no extra text).

Step 1 — Extract EXACTLY 3 Facts
Extract exactly 3 short factual claims from the political statement.
If fewer than 3 obvious facts exist, break broader claims into subfacts.
For fact1, fact2, fact3 output:
- fact text
- fact_label ∈ ["low", "medium", "high"]

Step 2 — Choose ONE Fact
Choose the best fact from fact1, fact2, fact3.
chosen_fact MUST equal one of fact1, fact2, fact3.
Criteria:
- highest fact_label (high > medium > low)
- if tied, pick most central to statement meaning

Output:
- chosen_fact
- chosen_fact_label
- chosen_fact_score (0–100)

Do NOT output any justification for fact1, fact2, fact3, or chosen_fact.

Step 3 — Statement Verification
Using the chosen fact:
Output:
- statement_verification_label ∈ ["Verified", "Likely false", "Uncertain"]
- statement_verification_justification (short sentence, <20 words)

Return ONLY this JSON object:

{
  "fact1": "",
  "fact1_label": "",
  "fact2": "",
  "fact2_label": "",
  "fact3": "",
  "fact3_label": "",
  "chosen_fact": "",
  "chosen_fact_label": "",
  "chosen_fact_score": 0,
  "statement_verification_label": "",
  "statement_verification_justification": ""
}
"""

# Include the speaker in the prompt
USER_TEMPLATE = """
You are given a political claim and who said it.

Speaker: {speaker}

Political statement:
"{statement}"

Follow the system instructions and return ONLY the JSON object.
"""


def build_messages(statement: str, speaker: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": USER_TEMPLATE.format(
                speaker=speaker if isinstance(speaker, str) else "",
                statement=statement
            )
        },
    ]


# ========================
# API CALL
# ========================

def call_openrouter(statement: str, speaker: str) -> str:
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://colab.research.google.com/",
        "X-Title": "P12 Mistral Parallel",
    }

    payload = {
        "model": MODEL_NAME,
        "messages": build_messages(statement, speaker),
        "temperature": 0.0,
        "max_tokens": 220,
    }

    resp = requests.post(OPENROUTER_URL, headers=headers, json=payload)
    if resp.status_code != 200:
        raise RuntimeError(f"Status {resp.status_code}: {resp.text[:200]}")

    return resp.json()["choices"][0]["message"]["content"]


# ========================
# JSON PARSER + NORMALIZER
# ========================

def parse_json(text: str) -> dict:
    """
    Try hard to turn the model output into a JSON dict.
    If it fails, return {} and let the caller handle it.
    """
    if not isinstance(text, str):
        return {}

    cleaned = text.strip()

    # Strip code fences like ```json ... ```
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        if len(lines) >= 3 and lines[0].startswith("```") and lines[-1].startswith("```"):
            cleaned = "\n".join(lines[1:-1]).strip()

    # Direct attempt
    try:
        return json.loads(cleaned)
    except:
        pass

    # Try between first { and last }
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1 and end > start:
        snippet = cleaned[start:end+1]
        try:
            return json.loads(snippet)
        except:
            pass

    return {}


def normalize(parsed: dict) -> dict:
    template = {
        "fact1": "",
        "fact1_label": "",
        "fact2": "",
        "fact2_label": "",
        "fact3": "",
        "fact3_label": "",
        "chosen_fact": "",
        "chosen_fact_label": "",
        "chosen_fact_score": 0,
        "statement_verification_label": "Uncertain",
        "statement_verification_justification": "",
    }

    for k in template:
        if isinstance(parsed, dict) and k in parsed:
            template[k] = parsed[k]

    # chosen_fact_score → int
    try:
        template["chosen_fact_score"] = int(float(template["chosen_fact_score"]))
    except:
        template["chosen_fact_score"] = 0

    def weight(lbl: str) -> int:
        return {"high": 3, "medium": 2, "low": 1}.get((lbl or "").lower(), 0)

    facts = [
        ("fact1", template["fact1"], template["fact1_label"]),
        ("fact2", template["fact2"], template["fact2_label"]),
        ("fact3", template["fact3"], template["fact3_label"]),
    ]

    # If all facts empty but chosen exists → copy chosen into all
    if all(f[1] == "" for f in facts) and template["chosen_fact"]:
        for name, _, _ in facts:
            template[name] = template["chosen_fact"]
        template["fact1_label"] = template["fact2_label"] = template["fact3_label"] = template["chosen_fact_label"]

    # If we have at least one non-empty fact, choose best
    if any(f[1] for f in facts):
        best = max(facts, key=lambda x: weight(x[2]))
        template["chosen_fact"] = best[1]
        template["chosen_fact_label"] = best[2]

    return template


# ========================
# WORKER FOR PARALLEL EXECUTION
# ========================

def process_one(row):
    stmt_id = row["id"]
    stmt_text = row["statement"]
    speaker = row.get("statement_originator", "")
    orig_label = row.get(
        "binary_label",
        row.get("original_true_false", row.get("original_verdict", ""))
    )

    try:
        raw = call_openrouter(stmt_text, speaker)
        parsed = parse_json(raw)

        if not parsed:
            norm = {
                "fact1": "",
                "fact1_label": "",
                "fact2": "",
                "fact2_label": "",
                "fact3": "",
                "fact3_label": "",
                "chosen_fact": "",
                "chosen_fact_label": "",
                "chosen_fact_score": 0,
                "statement_verification_label": "Uncertain",
                "statement_verification_justification": "JSON parse error from model output.",
            }
        else:
            norm = normalize(parsed)

    except Exception as e:
        norm = {
            "fact1": "",
            "fact1_label": "",
            "fact2": "",
            "fact2_label": "",
            "fact3": "",
            "fact3_label": "",
            "chosen_fact": "",
            "chosen_fact_label": "",
            "chosen_fact_score": 0,
            "statement_verification_label": "Uncertain",
            "statement_verification_justification": f"API error: {e}",
        }

    return stmt_id, {
        "id": stmt_id,
        "statement": stmt_text,
        "statement_originator": speaker,
        "model": MODEL_NAME,

        "fact1": norm["fact1"],
        "fact1_label": norm["fact1_label"],
        "fact2": norm["fact2"],
        "fact2_label": norm["fact2_label"],
        "fact3": norm["fact3"],
        "fact3_label": norm["fact3_label"],

        "chosen_fact": norm["chosen_fact"],
        "chosen_fact_label": norm["chosen_fact_label"],
        "chosen_fact_score": norm["chosen_fact_score"],

        "statement_verification_label": norm["statement_verification_label"],
        "statement_verification_justification": norm["statement_verification_justification"],

        "original_true_false": orig_label,
    }


# ========================
# MAIN PARALLEL EXECUTION
# ========================

df = pd.read_csv(INPUT_CSV)

# make sure required columns exist:
if "statement" not in df.columns:
    raise ValueError("INPUT_CSV must contain a 'statement' column.")
if "statement_originator" not in df.columns:
    raise ValueError("INPUT_CSV must contain a 'statement_originator' column for the speaker.")

if "id" not in df.columns:
    df["id"] = range(1, len(df) + 1)

df = df.head(NUM_STATEMENTS_TO_RUN)
rows = df.to_dict(orient="records")
total = len(rows)
results_dict = {}
completed = 0

print(f"Running {total} statements on Mistral with {MAX_WORKERS} workers...\n")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_one, row): row["id"] for row in rows}

    for future in as_completed(futures):
        stmt_id, output_row = future.result()
        results_dict[stmt_id] = output_row

        completed += 1
        pct = completed * 100 / total

        if completed % 10 == 0 or completed == total:
            print(f"[{completed}/{total}] {pct:.1f}% completed")

        if completed % BATCH_SIZE == 0 or completed == total:
            ordered = [results_dict[k] for k in sorted(results_dict.keys())]
            pd.DataFrame(ordered).to_csv(OUTPUT_CSV, index=False)
            print(f"  ↳ Saved {completed} rows to {OUTPUT_CSV}")

print("\nDONE!")
print(f"Saved final results to: {OUTPUT_CSV}")


Running 1050 statements on Mistral with 4 workers...

[10/1050] 1.0% completed
[20/1050] 1.9% completed
[30/1050] 2.9% completed
[40/1050] 3.8% completed
[50/1050] 4.8% completed
  ↳ Saved 50 rows to p12_mistral_with_author.csv
[60/1050] 5.7% completed
[70/1050] 6.7% completed
[80/1050] 7.6% completed
[90/1050] 8.6% completed
[100/1050] 9.5% completed
  ↳ Saved 100 rows to p12_mistral_with_author.csv
[110/1050] 10.5% completed
[120/1050] 11.4% completed
[130/1050] 12.4% completed
[140/1050] 13.3% completed
[150/1050] 14.3% completed
  ↳ Saved 150 rows to p12_mistral_with_author.csv
[160/1050] 15.2% completed
[170/1050] 16.2% completed
[180/1050] 17.1% completed
[190/1050] 18.1% completed
[200/1050] 19.0% completed
  ↳ Saved 200 rows to p12_mistral_with_author.csv
[210/1050] 20.0% completed
[220/1050] 21.0% completed
[230/1050] 21.9% completed
[240/1050] 22.9% completed
[250/1050] 23.8% completed
  ↳ Saved 250 rows to p12_mistral_with_author.csv
[260/1050] 24.8% completed
[270/1050] 25.